In [ ]:
import os
import urllib.request as ur
from pathlib import Path

import polars as pl
import xarray as xr

In [ ]:
url = "https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/goes18/l2/data/xrsf-l2-avg1m_science/sci_xrsf-l2-avg1m_g18_s20220617_e20260426_v2-2-1.nc"
data_dir = Path("../data/raw")
filename = "g18_xrsf_avg1m_backfill.nc"
path = data_dir / filename

if not os.path.exists(path):
    data_dir.mkdir(parents=True, exist_ok=True)

    ur.urlretrieve(url, path)

In [ ]:
ds = xr.open_dataset(path)
ds

In [ ]:
ds.isel(time=300)["xrsb_flag"].values

In [ ]:
df = pl.from_pandas(
    ds[["xrsb_flux", "xrsb_flag", "electron_correction_flag"]].to_pandas(),
    include_index=True,
)

In [ ]:
df

In [ ]:
df.select(pl.col("xrsb_flag").unique())

In [ ]:
df.select(pl.col("electron_correction_flag").unique())

In [ ]:
df.filter(
    ((pl.col("xrsb_flag").cast(pl.Int32) & 3) == 0)
    | ((pl.col("xrsb_flag").cast(pl.Int32) & 4) == 4)
)

In [ ]:
df.filter(
    ((pl.col("xrsb_flag").cast(pl.Int32) & 2) == 2)
    | ((pl.col("xrsb_flag").cast(pl.Int32) & 1) == 1)
)

In [ ]:
df.filter(
    ((pl.col("electron_correction_flag").cast(pl.Int32) & 1) == 1)
    | ((pl.col("electron_correction_flag").cast(pl.Int32) & 16) == 16)
)

In [ ]:
df.null_count()

In [ ]:
dfn = df.with_columns(
    pl.when(
        ((pl.col("xrsb_flag").cast(pl.Int32) & 2) == 2)
        | ((pl.col("xrsb_flag").cast(pl.Int32) & 1) == 1)
    )
    .then(None)
    .when(
        ((pl.col("electron_correction_flag").cast(pl.Int32) & 1) == 1)
        | ((pl.col("electron_correction_flag").cast(pl.Int32) & 16) == 16)
    )
    .then(None)
    .otherwise(pl.col("xrsb_flux"))
    .alias("xrsb_flux")
)

In [ ]:
dfn.null_count()

In [ ]:
dfn.select(pl.col("xrsb_flux").is_nan().sum())

In [ ]:
dfn

In [ ]:
dff = dfn.select(
    pl.col("time"), pl.col("xrsb_flux").fill_null(strategy="forward", limit=10)
)

In [ ]:
dff.null_count()

In [ ]:
dff.filter(pl.col("time") <= pl.col("time").shift(1))

In [ ]:
dff.with_columns(
    pl.col("time").diff().fill_null(pl.duration(minutes=1)).alias("gap")
).filter(pl.col("gap") > pl.duration(minutes=1))

In [ ]:
WINDOW_24H = 1440
WINDOW_12H = 720
MIN_FRAC = 0.8
MIN_24H = int(WINDOW_24H * MIN_FRAC)
MIN_12H = int(WINDOW_12H * MIN_FRAC)
C_CLASS_THRESHOLD = 1e-6

In [ ]:
df_feat = (
    dff.lazy()
    .with_columns(
        # Target
        pl.col("xrsb_flux")
        .rolling_max(WINDOW_24H, min_samples=MIN_24H)
        .shift(-WINDOW_24H)
        .alias("max_24h_la"),
        # Lag features
        pl.col("xrsb_flux").shift(15).alias("lag_15"),
        pl.col("xrsb_flux").shift(60).alias("lag_60"),
        pl.col("xrsb_flux").shift(120).alias("lag_120"),
        pl.col("xrsb_flux").shift(1440).alias("lag_1440"),
        # Rolling features
        pl.col("xrsb_flux")
        .rolling_max(WINDOW_12H, min_samples=MIN_12H)
        .alias("roll_max_720"),
        pl.col("xrsb_flux")
        .rolling_std(WINDOW_12H, min_samples=MIN_12H)
        .alias("roll_std_720"),
        pl.col("xrsb_flux")
        .rolling_mean(WINDOW_12H, min_samples=MIN_12H)
        .alias("roll_mean_720"),
        pl.col("xrsb_flux").diff(5).alias("deriv_1_5"),
        pl.col("xrsb_flux").diff(n=5).diff(n=5).alias("deriv_2_5"),
        (
            (pl.col("xrsb_flux") >= C_CLASS_THRESHOLD)
            & (pl.col("xrsb_flux").shift(1) < C_CLASS_THRESHOLD)
        )
        .rolling_sum(WINDOW_12H, min_samples=MIN_12H)
        .alias("roll_c_class_cross_720"),
    )
    .collect()
)

In [ ]:
df_feat

In [ ]:
df_feat.null_count()

In [ ]:
df_final = df_feat.drop_nulls()

In [ ]:
df_final

In [ ]:
df_final = df_final.with_columns(
    pl.col("time").dt.year().alias("year"),
    pl.col("time").dt.month().alias("month"),
)

In [ ]:
df_final.describe()

In [ ]:
lf = df.lazy()

# Null invalid flux values
lf = lf.with_columns(
    pl.when(
        ((pl.col("xrsb_flag").cast(pl.Int32) & 2) == 2)
        | ((pl.col("xrsb_flag").cast(pl.Int32) & 1) == 1)
    )
    .then(None)
    .when(
        ((pl.col("electron_correction_flag").cast(pl.Int32) & 1) == 1)
        | ((pl.col("electron_correction_flag").cast(pl.Int32) & 16) == 16)
    )
    .then(None)
    .otherwise(pl.col("xrsb_flux"))
    .alias("xrsb_flux")
)

# Forward fill null values
lf = lf.select(
    pl.col("time"), pl.col("xrsb_flux").fill_null(strategy="forward", limit=10)
)

# Feature extraction
lf = lf.with_columns(
    pl.col("xrsb_flux")
    .rolling_max(WINDOW_24H, min_samples=MIN_24H)
    .shift(-WINDOW_24H)
    .alias("max_24h_la"),
    # Lag features
    pl.col("xrsb_flux").shift(15).alias("lag_15"),
    pl.col("xrsb_flux").shift(60).alias("lag_60"),
    pl.col("xrsb_flux").shift(120).alias("lag_120"),
    pl.col("xrsb_flux").shift(1440).alias("lag_1440"),
    # Rolling features
    pl.col("xrsb_flux")
    .rolling_max(WINDOW_12H, min_samples=MIN_12H)
    .alias("roll_max_720"),
    pl.col("xrsb_flux")
    .rolling_std(WINDOW_12H, min_samples=MIN_12H)
    .alias("roll_std_720"),
    pl.col("xrsb_flux")
    .rolling_mean(WINDOW_12H, min_samples=MIN_12H)
    .alias("roll_mean_720"),
    pl.col("xrsb_flux").diff(5).alias("deriv_1_5"),
    pl.col("xrsb_flux").diff(n=5).diff(n=5).alias("deriv_2_5"),
    (
        (pl.col("xrsb_flux") >= C_CLASS_THRESHOLD)
        & (pl.col("xrsb_flux").shift(1) < C_CLASS_THRESHOLD)
    )
    .rolling_sum(WINDOW_12H, min_samples=MIN_12H)
    .alias("roll_c_class_cross_720"),
)

lf = lf.drop_nulls()

lf = lf.with_columns(
    pl.col("time").dt.year().alias("year"),
    pl.col("time").dt.month().alias("month"),
)
lf.collect().equals(df_final)